# On-policy distillation gradients: Qwen3 teacher → student

`test3.ipynb` ran this analysis on randomly initialized toy transformers. Their teacher and
student share nothing, and the gradient carried no usable signal. Here both models are
pretrained Qwen3 checkpoints with the same tokenizer:

| pair | teacher \(q\) (frozen) | student \(\pi_\theta\) |
| --- | --- | --- |
| A | Qwen3-4B | Qwen3-1.7B |
| B | Qwen3-8B | Qwen3-4B |

Prompts \(x\) are GSM8K train questions under the chat template. The student samples
\(y\sim\pi_\theta(\cdot\mid x)\) at temperature 1 with top-k and top-p off, for at most `T` tokens.
The objective is the full on-policy distillation loss

$$
J(\theta)=\mathbb{E}_{x}\,\mathbb{E}_{y\sim\pi_\theta(\cdot\mid x)}\sum_{t<|y|}
D_\alpha\big(\pi_\theta(\cdot\mid x,y_{<t}),\,q(\cdot\mid x,y_{<t})\big)
$$

\(D_\alpha\) is the full-vocabulary divergence of `compute_self_distillation_loss`
(`full_logit_distillation=True`), imported from `variance_study.token_divergence`: `0` forward
KL \(\mathrm{KL}(q\,\|\,\pi)\), `1` reverse KL, otherwise generalized JSD. Because the prefixes
are sampled from \(\pi_\theta\), the gradient has two parts:

$$
\nabla J=\mathbb{E}\Big[\underbrace{\sum_t\nabla D_t}_{\text{tokenwise}}
+\underbrace{\sum_t W_t\,\nabla\log\pi_\theta(y_t\mid x,y_{<t})}_{\text{score function}}\Big],
\qquad W_t=G_t-b_t,\quad G_t=\sum_{k>t}D_k
$$

The credit-weight rules below change \(W_t\) only.

Each pair runs three sections:

1. **Rollouts.** Lengths, per-position \(D_t\), and an on-policy check: the mean NLL of the
   sampled tokens should equal the mean policy entropy.
2. **Reference gradient.** `N_PROMPTS × N_COMPLETIONS` rollouts with two unbiased estimators,
   split-half cosines, and the tokenwise / score-function split by parameter group.
3. **Credit weights.** Per-sequence gradients on a separate rollout set, giving SNR and cosine
   to the reference for each rule.

Section 4 compares the two pairs.

**Gradient subset.** A flat fp32 gradient of a 1.7B model is 7 GB, and section 3 keeps one
accumulator per rule. Only `GRAD_MODULES` in the `GRAD_LAYER_FRACS` decoder layers are
unfrozen. The default is `o_proj` and `down_proj` in four layers from first to last: 67M
parameters for Qwen3-1.7B, 142M for Qwen3-4B. Every gradient here is restricted to that
subset. Backprop still runs through the whole network because layer 0 is included.

In [1]:
from __future__ import annotations

import gc
import json
import math
import re
import sys
import time
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig

CWD = Path.cwd().resolve()
ROOT = CWD if (CWD / "verl").is_dir() else CWD.parent
TOY_DIR = ROOT / "tests" / "special_standalone" / "future_returns_variance"
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(TOY_DIR))

from variance_study import divergence_name, future_returns, token_divergence  # noqa: E402
from verl.trainer.ppo.core_algos import (  # noqa: E402
    future_returns_token_scale,
    masked_group_leave_one_out_baseline,
    masked_leave_one_out_baseline,
)

PAIR_COLORS = ["#2a78d6", "#eb6834"]  # one fixed color per teacher/student pair
PIECE_COLORS = {"tokenwise": "#2a78d6", "score-function": "#eb6834"}

/home/hg22723/anaconda3/envs/sdql/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-11-05 08:55:04,893	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [2]:
# Rollouts and scoring. Rows are prompt-major: every completion of prompt 0, then prompt 1, ...


@dataclass
class Rollouts:
    prompt_ids: list[torch.Tensor]
    response_ids: list[torch.Tensor]  # sampled tokens, up to and including EOS
    prompt_index: np.ndarray  # [N] prompt id of each row; the group for the LOO baseline
    T: int  # max_new_tokens, the horizon of J

    def __len__(self) -> int:
        return len(self.response_ids)

    @property
    def lengths(self) -> torch.Tensor:
        return torch.tensor([len(r) for r in self.response_ids])

    def mask(self) -> torch.Tensor:
        return (torch.arange(self.T)[None] < self.lengths[:, None]).float()


def resolve_devices(student_device, teacher_device):
    """Explicit devices win. Otherwise the student gets the freest GPU and the teacher the next."""
    if not torch.cuda.is_available():
        return torch.device("cpu"), torch.device("cpu")
    order = sorted(
        range(torch.cuda.device_count()), key=lambda i: torch.cuda.mem_get_info(i)[0], reverse=True
    )
    student_device = student_device or f"cuda:{order[0]}"
    teacher_device = teacher_device or f"cuda:{order[1] if len(order) > 1 else order[0]}"
    return torch.device(student_device), torch.device(teacher_device)


def host_copy(x: torch.Tensor, device: torch.device) -> torch.Tensor:
    """Move x to `device` through host memory, never GPU-to-GPU directly.

    On this machine's PCIe A100s a direct peer copy returns corrupted data even though
    torch.cuda.can_device_access_peer is True, and the corruption appears to reach other
    memory on the destination GPU. Do not "test" peer copies on shared GPUs.
    """
    return x if x.device == device else x.cpu().to(device)


def load_frozen(name: str, device: torch.device, dtype: torch.dtype):
    model = AutoModelForCausalLM.from_pretrained(
        name, dtype=dtype, device_map={"": str(device)}, attn_implementation="sdpa"
    ).eval()
    for p in model.parameters():
        p.requires_grad_(False)
    return model


def load_prompts(path, n: int, seed: int, tokenizer, enable_thinking: bool) -> list[torch.Tensor]:
    df = pd.read_parquet(path)
    rows = np.random.default_rng(seed).choice(len(df), size=n, replace=False)
    prompts = []
    for r in rows:
        messages = [{"role": m["role"], "content": m["content"]} for m in df.iloc[int(r)]["prompt"]]
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=enable_thinking
        )
        prompts.append(torch.tensor(tokenizer(text, add_special_tokens=False)["input_ids"]))
    return prompts


def set_pure_sampling(model, tokenizer, max_new_tokens: int) -> list[int]:
    """Temperature 1 with top-k and top-p off, so samples come from pi_theta itself.

    This replaces model.generation_config rather than passing a config to generate(). Since
    transformers 4.50, generate() refills any passed field that equals the global default
    (temperature=1.0, top_p=1.0) from the model's config, which for Qwen3 is 0.6 / 0.95 / top-k 20.
    """
    eos = model.generation_config.eos_token_id
    eos = [eos] if isinstance(eos, int) else list(eos)
    model.generation_config = GenerationConfig(
        do_sample=True,
        temperature=1.0,
        top_k=0,
        top_p=1.0,
        max_new_tokens=max_new_tokens,
        eos_token_id=eos,
        pad_token_id=tokenizer.pad_token_id,
    )
    return eos


@torch.no_grad()
def sample(model, tokenizer, prompts, n_completions: int, T: int, batch_size: int, eos_ids) -> Rollouts:
    pad = tokenizer.pad_token_id
    eos = torch.tensor(eos_ids)
    rows = [(k, p) for k, p in enumerate(prompts) for _ in range(n_completions)]
    responses = []
    for lo in range(0, len(rows), batch_size):
        chunk = [p for _, p in rows[lo : lo + batch_size]]
        P = max(len(p) for p in chunk)
        ids = torch.full((len(chunk), P), pad, dtype=torch.long)
        attn = torch.zeros_like(ids)
        for j, p in enumerate(chunk):  # left-pad for batched generation
            ids[j, P - len(p) :] = p
            attn[j, P - len(p) :] = 1
        out = model.generate(input_ids=ids.to(model.device), attention_mask=attn.to(model.device))
        for g in out[:, P:].cpu():
            hit = torch.isin(g, eos).nonzero()
            end = int(hit[0, 0]) + 1 if len(hit) else len(g)
            responses.append(g[:end].clone())
    return Rollouts([p for _, p in rows], responses, np.array([k for k, _ in rows]), T)


def score(student, teacher, ro: Rollouts, rows, alpha: float, need_grad: bool) -> dict:
    """Per-token D_alpha and log pi(y_t) on `rows` of `ro`, aligned to response position t.

    Rows are right-padded, so causal attention keeps padding out of every real position and the
    default position ids are correct. The logits at column len(prompt) - 1 + t predict response
    token t; only those columns go through lm_head. Returns [B, Tb] tensors on the student device,
    where Tb is the longest response in `rows`. Entries past each response's end are 0.
    """
    sdev, tdev = student.device, teacher.device
    plen = torch.tensor([len(ro.prompt_ids[i]) for i in rows])
    rlen = torch.tensor([len(ro.response_ids[i]) for i in rows])
    B, Tb, Lmax = len(rows), int(rlen.max()), int((plen + rlen).max())
    ids = torch.zeros((B, Lmax), dtype=torch.long)
    attn = torch.zeros_like(ids)
    tok = torch.zeros((B, Tb), dtype=torch.long)
    for j, i in enumerate(rows):
        seq = torch.cat([ro.prompt_ids[i], ro.response_ids[i]])
        ids[j, : len(seq)] = seq
        attn[j, : len(seq)] = 1
        tok[j, : rlen[j]] = ro.response_ids[i]
    col = (plen[:, None] - 1 + torch.arange(Tb)[None]).clamp(max=Lmax - 1)
    row = torch.arange(B)[:, None]
    mask = (torch.arange(Tb)[None] < rlen[:, None]).float().to(sdev)

    def response_logits(model, dev):
        h = model.model(input_ids=ids.to(dev), attention_mask=attn.to(dev), use_cache=False).last_hidden_state
        return model.lm_head(h[row.to(dev), col.to(dev)])

    with torch.no_grad():  # bf16 logits cross devices through host memory; log_softmax runs in fp32
        t_logp = F.log_softmax(host_copy(response_logits(teacher, tdev), sdev).float(), dim=-1)
    with torch.set_grad_enabled(need_grad):
        s_logp = F.log_softmax(response_logits(student, sdev).float(), dim=-1)
        rewards = token_divergence(s_logp, t_logp.exp(), t_logp, alpha) * mask
        logp_a = s_logp.gather(-1, tok.to(sdev)[..., None]).squeeze(-1) * mask
    out = {"rewards": rewards, "logp_a": logp_a, "mask": mask}
    if not need_grad:
        out["entropy"] = -(s_logp.exp() * s_logp).sum(-1) * mask
    return out


@torch.no_grad()
def score_all(student, teacher, ro: Rollouts, alpha: float, micro_bs: int) -> dict:
    """No-grad pass: rewards r [N, T] (0 past the end), policy entropy and sampled-token NLL."""
    N, T = len(ro), ro.T
    out = {k: torch.zeros(N, T) for k in ("r", "entropy", "nll")}
    for lo in range(0, N, micro_bs):
        rows = list(range(lo, min(lo + micro_bs, N)))
        s = score(student, teacher, ro, rows, alpha, need_grad=False)
        Tb = s["rewards"].shape[1]
        out["r"][rows, :Tb] = s["rewards"].cpu()
        out["entropy"][rows, :Tb] = s["entropy"].cpu()
        out["nll"][rows, :Tb] = -s["logp_a"].cpu()
    out["mask"] = ro.mask()
    return out

In [3]:
# Gradients. Every loss is the per-sequence surrogate sum_t D_t + sum_t sg(W_t) log pi(y_t),
# summed over tokens and averaged over sequences, as in test2/test3.


def select_grad_params(model, layer_fracs, module_regex: str):
    """Unfreeze the matching weights in the chosen decoder layers. Everything else stays frozen.

    Returns (params, groups, layers). groups maps "L<layer> <module>" to its (lo, hi) span in the
    flat gradient.
    """
    n = model.config.num_hidden_layers
    layers = sorted({int(round(f * (n - 1))) for f in layer_fracs})
    pattern = re.compile(module_regex)
    params, groups, offset = [], {}, 0
    for name, p in model.named_parameters():
        m = re.fullmatch(r"model\.layers\.(\d+)\.(.+)", name)
        if m is None or int(m.group(1)) not in layers or not pattern.search(m.group(2)):
            continue
        p.requires_grad_(True)
        params.append(p)
        groups[f"L{int(m.group(1)):02d} {m.group(2).removesuffix('.weight')}"] = (offset, offset + p.numel())
        offset += p.numel()
    if not params:
        raise ValueError(f"no parameters matched layers={layers} regex={module_regex!r}")
    return params, groups, layers


def flat_grad(loss, params, retain_graph: bool) -> torch.Tensor:
    grads = torch.autograd.grad(loss, params, retain_graph=retain_graph)
    return torch.cat([g.reshape(-1).float() for g in grads])


def cosine(a: torch.Tensor, b: torch.Tensor) -> float:
    return F.cosine_similarity(a.flatten(), b.flatten(), dim=0).item()


def ceiling_from_split_half(c: float) -> float:
    """Estimated cos(full-sample gradient, true grad J) from the split-half cosine c.

    With isotropic noise, c = s / (s + n) for signal s and per-half noise n. The full sample has
    noise n / 2, so its cosine to the truth is sqrt(2c / (1 + c)).
    """
    return math.sqrt(2 * c / (1 + c)) if c > 0 else float("nan")


def heldout_baseline(r_base: torch.Tensor) -> dict:
    """Scalar b = mean return and per-position b_t = sum_{k>t} mean_i r_ik from held-out rollouts."""
    return {"R": r_base.sum(-1).mean().item(), "G": future_returns(r_base.mean(0, keepdim=True))}


# Whether the rule keeps the on-policy gradient unbiased. "O(1/N)": the batch mean includes the row
# itself, which shrinks the score term by (1 - 1/N). Rescaled rules change the score/tokenwise ratio,
# so they estimate a different vector.
UNBIASED = {
    "no PG term (tokenwise)": "no",
    "full trajectory R": "yes",
    "full trajectory R - b (held-out)": "yes",
    "G_t - b_t (held-out)": "yes",
    "raw G_t": "yes",
    "G_t - batch_mean_t": "O(1/N)",
    "G_t / remaining": "no",
    "(G_t - batch_mean_t)/remaining": "no",
    "G_t / (T-1)": "no",
    "(G_t - batch_mean_t)/(T-1)": "no",
    "G(r - batch LOO_t)": "yes",
    "G(r - group LOO_t)": "yes",
    "G(r - group LOO_t)/remaining_i": "no",
    "G(r - group LOO_t)/seq_len_i": "no",
}


def credit_weights(r: torch.Tensor, mask: torch.Tensor, index: np.ndarray, base: dict) -> dict:
    """W_t [N, T] for every rule in UNBIASED, from detached rewards r (0 past each end).

    The test3 rules use plain batch means over all rows, so G_t - mean_t equals G(r - mean) at
    gamma = 1. The LOO rules are core_algos' compute_self_distillation_loss: a masked leave-one-out
    mean of r per position (batch or per-prompt group), then G, then the optional
    future_returns_reward_scale divisor (remaining_i = (T_i - t).clamp(1), seq_len_i = T_i).
    """
    T = r.shape[1]
    G = future_returns(r)
    Gc = G - G.mean(0, keepdim=True)
    rem = (T - torch.arange(T) - 1).clamp(min=1).to(r)  # fixed-horizon remaining, as in test3
    g_loo = future_returns((r - masked_group_leave_one_out_baseline(r, mask, index)) * mask)
    b_loo = future_returns((r - masked_leave_one_out_baseline(r, mask)) * mask)
    R = r.sum(-1, keepdim=True).expand_as(r)
    W = {
        "no PG term (tokenwise)": torch.zeros_like(r),
        "full trajectory R": R,
        "full trajectory R - b (held-out)": R - base["R"],
        "G_t - b_t (held-out)": G - base["G"],
        "raw G_t": G,
        "G_t - batch_mean_t": Gc,
        "G_t / remaining": G / rem,
        "(G_t - batch_mean_t)/remaining": Gc / rem,
        "G_t / (T-1)": G / (T - 1),
        "(G_t - batch_mean_t)/(T-1)": Gc / (T - 1),
        "G(r - batch LOO_t)": b_loo,
        "G(r - group LOO_t)": g_loo,
        "G(r - group LOO_t)/remaining_i": g_loo / future_returns_token_scale(mask, "remaining"),
        "G(r - group LOO_t)/seq_len_i": g_loo / future_returns_token_scale(mask, "seq_len"),
    }
    assert list(W) == list(UNBIASED)
    return W


def batch_gradient(student, teacher, params, ro: Rollouts, W: torch.Tensor, alpha: float, micro_bs: int):
    """Mean over rows of grad[sum_t D_t + sum_t W_t log pi(y_t)], computed in padded micro-batches."""
    acc = None
    for lo in range(0, len(ro), micro_bs):
        rows = list(range(lo, min(lo + micro_bs, len(ro))))
        s = score(student, teacher, ro, rows, alpha, need_grad=True)
        Tb = s["rewards"].shape[1]
        loss = s["rewards"].sum() + (W[rows, :Tb].to(s["logp_a"].device) * s["logp_a"]).sum()
        g = flat_grad(loss, params, retain_graph=False)
        acc = g if acc is None else acc + g
    return acc / len(ro)


def reference_gradient(student, teacher, params, groups, ro, sc, base, alpha, micro_bs, ref_key):
    """Large-sample on-policy gradient on `ro`, accumulated separately per half of the prompts.

    Pieces: tokenwise sum_t grad D_t; whole-trajectory score (R - b) grad log pi(y) with held-out
    b ("traj"); reward-to-go score G(r - group LOO)_t grad log pi(y_t), the training estimator
    ("rtg"). Each on-policy estimator is tokenwise + one score piece, and both are unbiased.
    """
    dev = student.device
    r, mask, idx = sc["r"], sc["mask"], ro.prompt_index
    weights = {
        "traj": (r.sum(-1, keepdim=True) - base["R"]).expand_as(r),
        "rtg": future_returns((r - masked_group_leave_one_out_baseline(r, mask, idx)) * mask),
    }
    half = (idx >= (idx.max() + 1) // 2).astype(int)
    n_half = np.bincount(half, minlength=2)
    P = sum(p.numel() for p in params)
    acc = {(k, h): torch.zeros(P, device=dev) for k in ("tok", "traj", "rtg") for h in (0, 1)}
    for h in (0, 1):
        rows_h = np.flatnonzero(half == h)
        for lo in range(0, len(rows_h), micro_bs):
            rows = rows_h[lo : lo + micro_bs].tolist()
            s = score(student, teacher, ro, rows, alpha, need_grad=True)
            Tb = s["rewards"].shape[1]
            losses = {"tok": s["rewards"].sum()}
            for k, w in weights.items():
                losses[k] = (w[rows, :Tb].to(dev) * s["logp_a"]).sum()
            for j, (k, loss) in enumerate(losses.items()):
                acc[(k, h)] += flat_grad(loss, params, retain_graph=j < len(losses) - 1) / n_half[h]
            del s, losses

    w0, w1 = n_half / n_half.sum()
    full = lambda key: w0 * acc[(key, 0)] + w1 * acc[(key, 1)]  # noqa: E731
    est_rows, piece_rows, vecs = [], [], {"tok": full("tok")}
    for k in ("traj", "rtg"):
        a, b = acc[("tok", 0)] + acc[(k, 0)], acc[("tok", 1)] + acc[(k, 1)]
        vecs[k] = w0 * a + w1 * b
        c = cosine(a, b)
        est_rows.append({
            "estimator": {"traj": "R - b (held-out), whole trajectory", "rtg": "G(r - group LOO), reward-to-go"}[k],
            "key": k,
            "norm": vecs[k].norm().item(),
            "split_half_cos": c,
            "est_cos_to_true": ceiling_from_split_half(c),
        })
    est_df = pd.DataFrame(est_rows)
    est_df["cos_traj_vs_rtg"] = cosine(vecs["traj"], vecs["rtg"])

    g = vecs[ref_key]
    for name, key in (("tokenwise", "tok"), ("score-function", ref_key)):
        piece_rows.append({
            "piece": name,
            "norm": full(key).norm().item(),
            "split_half_cos": cosine(acc[(key, 0)], acc[(key, 1)]),
            "cos_to_onpolicy": cosine(full(key), g),
        })
    piece_rows.append({"piece": "on-policy", "norm": g.norm().item(), "cos_to_onpolicy": 1.0,
                       "split_half_cos": est_df.set_index("key").loc[ref_key, "split_half_cos"]})
    piece_df = pd.DataFrame(piece_rows)

    tok, pg = vecs["tok"], g - vecs["tok"]
    n_layers = student.config.num_hidden_layers
    group_rows = []
    for key, (lo, hi) in groups.items():
        layer, module = key.split(" ", 1)
        group_rows.append({
            "group": key,
            "depth": int(layer[1:]) / (n_layers - 1),
            "module": module,
            "params": hi - lo,
            "token_norm": tok[lo:hi].norm().item(),
            "pg_norm": pg[lo:hi].norm().item(),
            "onpolicy_norm": g[lo:hi].norm().item(),
            "cos_token_onpolicy": cosine(tok[lo:hi], g[lo:hi]),
            "cos_pg_onpolicy": cosine(pg[lo:hi], g[lo:hi]),
            "pg_split_half_cos": cosine(acc[(ref_key, 0)][lo:hi], acc[(ref_key, 1)][lo:hi]),
        })
    del acc
    return vecs, est_df, piece_df, pd.DataFrame(group_rows)


def credit_metrics(student, teacher, params, groups, ro, sc, base, alpha, g_ref, ref_ceiling, micro_bs):
    """Per-sequence gradients for every credit rule; each row summarizes their mean against g_ref.

    Streams the per-sequence grads: one fp32 sum vector and one sum of squared norms per rule, so
    noise^2 = mean_i ||g_i||^2 - ||mean||^2 (test3's grad_metrics) without storing [N, P].
    """
    dev = student.device
    W = credit_weights(sc["r"], sc["mask"], ro.prompt_index, base)
    names = list(W)
    s1 = {k: torch.zeros_like(g_ref) for k in names}
    s2 = dict.fromkeys(names, 0.0)
    for i in range(len(ro)):
        s = score(student, teacher, ro, [i], alpha, need_grad=True)
        L = s["rewards"].shape[1]
        direct = s["rewards"].sum()
        for j, k in enumerate(names):
            loss = direct + (W[k][i, :L].to(dev) * s["logp_a"][0]).sum()
            g = flat_grad(loss, params, retain_graph=j < len(names) - 1)
            s1[k] += g
            s2[k] += torch.dot(g, g).item()
        del s, direct

    N = len(ro)
    rows, group_rows = [], []
    for k in names:
        mean = s1[k] / N
        mean_sq = torch.dot(mean, mean).item()
        noise = math.sqrt(max(s2[k] / N - mean_sq, 0.0))
        c = cosine(mean, g_ref)
        rows.append({
            "variant": k,
            "unbiased": UNBIASED[k],
            "snr": math.sqrt(mean_sq) / noise if noise > 0 else float("nan"),
            "cos": c,
            "cos_corrected": c / ref_ceiling,
            "mean_norm": math.sqrt(mean_sq),
            "noise": noise,
        })
        group_rows.append({"variant": k, **{gk: cosine(mean[lo:hi], g_ref[lo:hi]) for gk, (lo, hi) in groups.items()}})

    # Port check: the per-sequence mean must equal one padded micro-batch gradient of the same loss.
    check = "G(r - group LOO_t)"
    g_batch = batch_gradient(student, teacher, params, ro, W[check], alpha, micro_bs)
    rel = ((s1[check] / N - g_batch).norm() / g_batch.norm()).item()
    del s1
    return pd.DataFrame(rows), pd.DataFrame(group_rows).set_index("variant"), rel

In [4]:
# One teacher/student pair, end to end. Results go to RESULTS_DIR/<slug>/ as CSV and JSON.


def masked_mean_std(x: torch.Tensor, mask: torch.Tensor):
    n = mask.sum(0).clamp(min=1)
    mean = (x * mask).sum(0) / n
    std = (((x - mean) ** 2) * mask).sum(0).div(n).sqrt()
    return mean.numpy(), std.numpy(), mask.mean(0).numpy()


def rollout_row(name: str, ro: Rollouts, sc: dict, eos_ids) -> dict:
    m = sc["mask"]
    n_tok = m.sum()
    R = sc["r"].sum(-1)
    return {
        "set": name,
        "sequences": len(ro),
        "mean_len": ro.lengths.float().mean().item(),
        "truncated": float(np.mean([int(r[-1]) not in eos_ids for r in ro.response_ids])),
        "mean_D_per_token": (sc["r"].sum() / n_tok).item(),
        "mean_R": R.mean().item(),
        "std_R": R.std().item(),
        "sampled_NLL": (sc["nll"].sum() / n_tok).item(),
        "entropy": (sc["entropy"].sum() / n_tok).item(),
    }


def style_axis(ax):
    ax.grid(axis="x", color="#e4e3df", lw=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)


def analyze_pair(teacher_name: str, student_name: str, color: str) -> dict:
    tag = f"{teacher_name.split('/')[-1]} -> {student_name.split('/')[-1]}"
    slug = re.sub(r"[^a-z0-9.]+", "_", tag.lower()).strip("_")
    out_dir = RESULTS_DIR / slug
    out_dir.mkdir(parents=True, exist_ok=True)
    clock = time.time()

    def log(msg):
        print(f"[{tag} | {time.time() - clock:5.0f}s] {msg}", flush=True)

    sdev, tdev = resolve_devices(STUDENT_DEVICE, TEACHER_DEVICE)
    tokenizer = AutoTokenizer.from_pretrained(student_name)
    if AutoTokenizer.from_pretrained(teacher_name).get_vocab() != tokenizer.get_vocab():
        raise ValueError(f"{teacher_name} and {student_name} use different tokenizers")
    student = load_frozen(student_name, sdev, STUDENT_DTYPE)
    teacher = load_frozen(teacher_name, tdev, TEACHER_DTYPE)
    if student.lm_head.out_features != teacher.lm_head.out_features:
        raise ValueError("teacher and student vocab sizes differ")
    eos_ids = set_pure_sampling(student, tokenizer, T)
    params, groups, layers = select_grad_params(student, GRAD_LAYER_FRACS, GRAD_MODULES)
    n_grad = sum(p.numel() for p in params)
    log(
        f"student {sdev}, teacher {tdev}, {divergence_name(ALPHA)}. Grad subset: layers {layers} of "
        f"{student.config.num_hidden_layers}, {n_grad / 1e6:.1f}M params, "
        f"~{(len(UNBIASED) + 1) * n_grad * 4 / 2**30:.1f} GiB of accumulators in section 3"
    )

    # ---- 1. Rollouts ----
    prompts = load_prompts(PROMPT_PARQUET, N_PROMPTS + N_METRIC_PROMPTS + N_BASE, SEED, tokenizer, ENABLE_THINKING)
    torch.manual_seed(SEED)
    splits = {
        "reference": (prompts[:N_PROMPTS], N_COMPLETIONS),
        "metric": (prompts[N_PROMPTS : N_PROMPTS + N_METRIC_PROMPTS], N_METRIC_COMPLETIONS),
        "held-out baseline": (prompts[N_PROMPTS + N_METRIC_PROMPTS :], 1),
    }
    sets = {k: sample(student, tokenizer, p, n, T, GEN_BS, eos_ids) for k, (p, n) in splits.items()}
    log(f"sampled {sum(len(s) for s in sets.values())} rollouts")
    scores = {k: score_all(student, teacher, ro, ALPHA, MICRO_BS) for k, ro in sets.items()}
    base = heldout_baseline(scores["held-out baseline"]["r"])
    rollout_df = pd.DataFrame([rollout_row(k, sets[k], scores[k], eos_ids) for k in sets])
    log("scored rollouts. sampled_NLL should match entropy if sampling is on-policy")
    display(rollout_df.round(3))
    ex = sets["reference"]
    print("example response:\n" + tokenizer.decode(ex.response_ids[0])[:800] + "\n")

    mean, std, alive = masked_mean_std(scores["reference"]["r"], scores["reference"]["mask"])
    t = np.arange(T)
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
    axes[0].plot(t, mean, lw=2, color=color)
    axes[0].fill_between(t, mean - std, mean + std, color=color, alpha=0.15, lw=0)
    axes[0].set(xlabel="response position t", ylabel=r"$D_\alpha$ over unfinished rows",
                title=f"{tag}: per-position divergence (mean ± 1 sd)")
    axes[1].plot(t, alive, lw=2, color=color)
    axes[1].set(xlabel="response position t", ylabel="fraction still generating", ylim=(0, 1.02),
                title="Rollouts still running")
    for ax in axes:
        style_axis(ax)
    fig.tight_layout()
    plt.show()

    # ---- 2. Reference gradient ----
    vecs, est_df, piece_df, group_df = reference_gradient(
        student, teacher, params, groups, sets["reference"], scores["reference"], base, ALPHA, MICRO_BS, REF_ESTIMATOR
    )
    ref_ceiling = est_df.set_index("key").loc[REF_ESTIMATOR, "est_cos_to_true"]
    log(f"reference gradient on {len(sets['reference'])} rollouts; using '{REF_ESTIMATOR}'")
    display(est_df.round(4))
    display(piece_df.round(4))
    display(group_df.drop(columns=["depth", "module"]).round(4))

    fig, ax = plt.subplots(figsize=(8.5, 0.32 * len(group_df) + 1.4))
    y = np.arange(len(group_df))
    for k, (piece, col) in enumerate((("tokenwise", "cos_token_onpolicy"), ("score-function", "cos_pg_onpolicy"))):
        ax.barh(y + (k - 0.5) * 0.38, group_df[col], height=0.36, color=PIECE_COLORS[piece], label=piece)
    ax.set_yticks(y, group_df["group"])
    ax.invert_yaxis()
    ax.axvline(0, color="#8a8984", lw=0.8)
    ax.set(xlabel="cosine to the on-policy reference", title=f"{tag}: which piece carries the gradient")
    fig.legend(*ax.get_legend_handles_labels(), loc="lower center", ncol=2, fontsize=8, frameon=False)
    style_axis(ax)
    fig.tight_layout(rect=(0, 0.06, 1, 1))
    plt.show()

    g_ref = vecs[REF_ESTIMATOR]
    del vecs
    torch.cuda.empty_cache()

    # ---- 3. Credit weights ----
    credit_df, credit_groups, port_rel = credit_metrics(
        student, teacher, params, groups, sets["metric"], scores["metric"], base, ALPHA, g_ref, ref_ceiling, MICRO_BS
    )
    log(
        f"credit weights on {len(sets['metric'])} rollouts. Per-sequence mean vs padded micro-batch "
        f"gradient: rel err {port_rel:.1e} (fp32 gives ~2e-5, bf16 ~1e-1; misaligned tokens would be ~1)"
    )
    display(credit_df.round(4))
    display(credit_groups.round(3))

    fig, axes = plt.subplots(1, 2, figsize=(13, 0.34 * len(credit_df) + 1.6), sharey=True)
    order = list(credit_df["variant"])
    axes[0].barh(order, credit_df["snr"], color=color, height=0.6)
    axes[0].set(xlabel="SNR = ||mean|| / per-sequence noise")
    axes[1].barh(order, credit_df["cos"], color=color, height=0.6)
    axes[1].axvline(ref_ceiling, color="#52514e", lw=1, ls="--", label=f"reference ceiling ≈ {ref_ceiling:.2f}")
    axes[1].axvline(0, color="#8a8984", lw=0.8)
    axes[1].set(xlabel="cosine to on-policy reference")
    axes[1].legend(loc="lower right", fontsize=8, frameon=False)
    axes[0].invert_yaxis()
    for ax in axes:
        style_axis(ax)
    fig.suptitle(f"{tag}: credit weights, N={len(sets['metric'])}, T={T}, {divergence_name(ALPHA)}")
    fig.tight_layout()
    plt.show()

    res = {
        "tag": tag,
        "rollouts": rollout_df,
        "curve": pd.DataFrame({"t": t, "mean_D": mean, "std_D": std, "alive": alive}),
        "estimators": est_df,
        "pieces": piece_df,
        "groups": group_df,
        "credit": credit_df,
        "credit_groups": credit_groups,
    }
    for k, v in res.items():
        if isinstance(v, pd.DataFrame):
            v.to_csv(out_dir / f"{k}.csv", index=(k == "credit_groups"))
    config = {k: str(v) for k, v in KNOBS.items()} | {
        "teacher": teacher_name, "student": student_name, "layers": layers, "grad_params": n_grad,
        "port_rel_err": port_rel, "seconds": time.time() - clock,
    }
    (out_dir / "config.json").write_text(json.dumps(config, indent=2))
    log(f"saved to {out_dir}")

    del student, teacher, params, g_ref
    gc.collect()
    torch.cuda.empty_cache()
    return res


def load_results(out_dir: Path) -> dict:
    config = json.loads((out_dir / "config.json").read_text())
    res = {"tag": f"{config['teacher'].split('/')[-1]} -> {config['student'].split('/')[-1]}"}
    for f in out_dir.glob("*.csv"):
        res[f.stem] = pd.read_csv(f, index_col=0 if f.stem == "credit_groups" else None)
    return res

## Knobs

Both pairs use the same prompts and seed. Rollout sets:

- **reference**: `N_PROMPTS × N_COMPLETIONS`, for the reference gradient
- **metric**: `N_METRIC_PROMPTS × N_METRIC_COMPLETIONS`, for the per-sequence credit-weight metrics
  (≥ 2 completions per prompt, so the group LOO baseline exists)
- **held-out baseline**: `N_BASE` rollouts, one per prompt, for the scalar \(b\) and per-position \(b_t\)

Memory. The student and teacher go on the two GPUs with the most free memory, unless
`STUDENT_DEVICE` / `TEACHER_DEVICE` are set. Both default to fp32. In bf16, the per-sequence vs
padded-batch port check is off by 6–10%, which is too much for comparing gradient cosines; in
fp32 it agrees to about 2e-5. For pair B in fp32:

- teacher GPU: about 32 GB for Qwen3-8B
- student GPU: about 16 GB of Qwen3-4B weights, about 8 GB of section-3 accumulators with the
  default subset, plus activations

bf16 halves the weights if memory is short. Lower `MICRO_BS` or `T` if the full-vocabulary
logits (`MICRO_BS × T × 151936` floats) do not fit.

**Do not copy tensors directly between GPUs on this machine.** Direct peer copies between its
PCIe A100s return corrupted data. `score()` therefore sends the teacher's logits through host
memory (`host_copy`).

In [5]:
PAIRS = [  # (teacher, student)
    ("Qwen/Qwen3-4B", "Qwen/Qwen3-1.7B"),
    ("Qwen/Qwen3-8B", "Qwen/Qwen3-4B"),
]
PROMPT_PARQUET = ROOT / "datasets" / "new" / "gsm8k" / "train.parquet"
ENABLE_THINKING = False  # Qwen3 chat template; True gives long <think> traces truncated at T
T = 256  # max new tokens: the horizon of J
ALPHA = 0.0  # 0=forward KL, 1=reverse KL, 0.5=JSD
SEED = 0
N_PROMPTS = 64  # reference gradient: N_PROMPTS x N_COMPLETIONS rollouts
N_COMPLETIONS = 8
N_METRIC_PROMPTS = 8  # credit metrics: N_METRIC_PROMPTS x N_METRIC_COMPLETIONS rollouts
N_METRIC_COMPLETIONS = 4
N_BASE = 128  # held-out rollouts for b and b_t
GEN_BS = 32  # generation batch size
MICRO_BS = 2  # scoring / gradient micro-batch
# fp32: the per-sequence vs padded-batch port check agrees to ~2e-5. With bf16 it is off by 6-10%,
# mostly from the teacher, because the tokenwise gradient scales with (pi - q) and pi ~ q.
STUDENT_DTYPE = torch.float32
TEACHER_DTYPE = torch.float32
STUDENT_DEVICE = None  # e.g. "cuda:0"; None picks the freest GPU
TEACHER_DEVICE = None  # None picks the next freest GPU
GRAD_LAYER_FRACS = (0.0, 1 / 3, 2 / 3, 1.0)  # decoder layers at these depths are unfrozen
GRAD_MODULES = r"(self_attn\.o_proj|mlp\.down_proj)\.weight$"
REF_ESTIMATOR = "rtg"  # "rtg" = G(r - group LOO), "traj" = whole-trajectory R - b
RESULTS_DIR = ROOT / "notebook" / "test_4_results"

KNOBS = {k: v for k, v in globals().items() if k.isupper() and k not in ("PAIR_COLORS", "PIECE_COLORS", "UNBIASED", "KNOBS")}
RESULTS: dict[str, dict] = {}
print(f"T={T}  {divergence_name(ALPHA)}  reference={N_PROMPTS}x{N_COMPLETIONS}  "
      f"metric={N_METRIC_PROMPTS}x{N_METRIC_COMPLETIONS}  base={N_BASE}  thinking={ENABLE_THINKING}")

T=256  forward_kl  reference=64x8  metric=8x4  base=128  thinking=False


## How to read each pair's output

**1. Rollouts.** `sampled_NLL` and `entropy` should agree to within sampling noise. If NLL is
clearly below entropy, generation was not pure sampling, and the score-function term is biased.
The curve is the mean \(D_t\) over rows that are still generating.

**2. Reference gradient.** Two unbiased estimators run on the same reference rollouts.

- `traj`: \(\sum_t\nabla D_t+(R-b)\sum_t\nabla\log\pi_t\), with held-out scalar \(b\)
- `rtg`: \(\sum_t\nabla D_t+\sum_t G(r-\mathrm{LOO})_t\,\nabla\log\pi_t\), the training estimator,
  where the per-position LOO mean is taken over the other completions of the same prompt

`split_half_cos` compares the gradients of the two prompt halves. `est_cos_to_true` converts it
into the expected cosine between the full-sample gradient and the true \(\nabla J\):
\(\sqrt{2c/(1+c)}\), assuming isotropic noise. That value is the ceiling for every cosine in
section 3. The two estimators share rollouts and the tokenwise piece, so `cos_traj_vs_rtg` is
only a loose consistency check, not an independent one.

The piece table splits the reference into its tokenwise and score-function parts. A
score-function `split_half_cos` near 0 means the score-function term is still noise at this
sample size. The group table and bar chart show the same split for each unfrozen weight matrix.

**3. Credit weights.** Each row is the mean of `N_METRIC_PROMPTS × N_METRIC_COMPLETIONS`
per-sequence gradients for one \(W_t\) rule, compared with the reference. SNR and noise follow
test3's `grad_metrics`. `cos_corrected = cos / ceiling` estimates the cosine to the true
\(\nabla J\), with the reference's own noise removed. `unbiased` marks which rules estimate
\(\nabla J\) itself. The port check compares the per-sequence mean with one padded micro-batch
gradient of the same loss.

## Pair A: Qwen3-4B teacher → Qwen3-1.7B student

In [6]:
teacher_name, student_name = PAIRS[0]
RESULTS["A"] = analyze_pair(teacher_name, student_name, PAIR_COLORS[0])

KeyboardInterrupt: 

## Pair B: Qwen3-8B teacher → Qwen3-4B student

In [ ]:
teacher_name, student_name = PAIRS[1]
RESULTS["B"] = analyze_pair(teacher_name, student_name, PAIR_COLORS[1])

## 4. Pair comparison

This section uses the in-memory `RESULTS`. After a kernel restart it reloads them from
`RESULTS_DIR`. Layer indices differ between the models (28 vs 36 layers), so the per-group plot
uses relative depth.

In [ ]:
if not RESULTS:
    RESULTS = {d.name: load_results(d) for d in sorted(RESULTS_DIR.iterdir()) if (d / "config.json").exists()}
runs = list(RESULTS.values())


def summary_row(res: dict) -> dict:
    ro = res["rollouts"].set_index("set").loc["reference"]
    est = res["estimators"].set_index("key")
    pieces = res["pieces"].set_index("piece")
    return {
        "pair": res["tag"],
        "mean_len": ro["mean_len"],
        "truncated": ro["truncated"],
        "mean_D_per_token": ro["mean_D_per_token"],
        "mean_R": ro["mean_R"],
        "NLL - entropy": ro["sampled_NLL"] - ro["entropy"],
        "split_half (rtg)": est.loc["rtg", "split_half_cos"],
        "split_half (traj)": est.loc["traj", "split_half_cos"],
        "ceiling (ref)": est.loc[REF_ESTIMATOR, "est_cos_to_true"],
        "||tokenwise||": pieces.loc["tokenwise", "norm"],
        "||score||": pieces.loc["score-function", "norm"],
        "score split_half": pieces.loc["score-function", "split_half_cos"],
        "cos(tokenwise, on-policy)": pieces.loc["tokenwise", "cos_to_onpolicy"],
    }


display(pd.DataFrame([summary_row(r) for r in runs]).set_index("pair").T.round(4))

# Credit-weight SNR and cosine, one bar per pair.
variants = list(runs[0]["credit"]["variant"])
y = np.arange(len(variants))
h = 0.8 / len(runs)
fig, axes = plt.subplots(1, 2, figsize=(13.5, 0.42 * len(variants) + 1.8), sharey=True)
for k, res in enumerate(runs):
    cdf = res["credit"].set_index("variant").loc[variants]
    ceiling = res["estimators"].set_index("key").loc[REF_ESTIMATOR, "est_cos_to_true"]
    offset = (k - (len(runs) - 1) / 2) * h
    axes[0].barh(y + offset, cdf["snr"], height=h * 0.92, color=PAIR_COLORS[k], label=res["tag"])
    axes[1].barh(y + offset, cdf["cos"], height=h * 0.92, color=PAIR_COLORS[k])
    axes[1].axvline(ceiling, color=PAIR_COLORS[k], lw=1.2, ls="--", label=f"{res['tag']} reference ceiling ≈ {ceiling:.2f}")
axes[0].set(xlabel="SNR", yticks=y, yticklabels=variants)
axes[1].set(xlabel="cosine to that pair's on-policy reference")
axes[1].axvline(0, color="#8a8984", lw=0.8)
axes[0].invert_yaxis()
for ax in axes:
    style_axis(ax)
handles = [hd for ax in axes for hd in ax.get_legend_handles_labels()[0]]
labels = [lb for ax in axes for lb in ax.get_legend_handles_labels()[1]]
fig.legend(handles, labels, loc="lower center", ncol=2, fontsize=8, frameon=False)
fig.suptitle(f"Credit weights by pair  T={T}  {divergence_name(ALPHA)}")
fig.tight_layout(rect=(0, 0.07, 1, 1))
plt.show()

# Per-position divergence and the tokenwise share of the gradient by depth.
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for k, res in enumerate(runs):
    c = res["curve"]
    axes[0].plot(c["t"], c["mean_D"], lw=2, color=PAIR_COLORS[k], label=res["tag"])
    g = res["groups"]
    for module, marker in zip(sorted(g["module"].unique()), ("o", "s", "^", "D")):
        sub = g[g["module"] == module].sort_values("depth")
        axes[1].plot(sub["depth"], sub["cos_token_onpolicy"], lw=2, marker=marker, ms=8,
                     color=PAIR_COLORS[k], label=f"{res['tag']}  {module}")
axes[0].set(xlabel="response position t", ylabel=r"mean $D_\alpha$ over unfinished rows",
            title="Per-position divergence")
axes[1].set(xlabel="relative layer depth", ylabel="cos(tokenwise, on-policy reference)",
            title="Tokenwise share of the gradient by depth")
for ax in axes:
    ax.legend(fontsize=8, frameon=False)
    style_axis(ax)
fig.tight_layout()
plt.show()